# Jacobians and Hessians

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 02.05 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/06_jacobians_and_hessians.ipynb)

---

## 🎯 Learning Objective

Generalize the derivative to vector-valued functions (Jacobians) and second derivatives (Hessians), and use Taylor expansion to locally approximate a loss surface.

---

## 📐 Theory

So far, derivatives have mapped a function to a number ($f'(a)$) or a vector ($\nabla f(a)$).
This notebook covers the two remaining generalizations: derivatives of *vector-valued*
functions, and *second* derivatives of scalar functions with many inputs.

### The Jacobian matrix

If $\mathbf{f}:\mathbb{R}^n \to \mathbb{R}^m$ outputs a vector
$\mathbf{f}(\mathbf{x}) = (f_1(\mathbf{x}), \dots, f_m(\mathbf{x}))$, its **Jacobian** collects
every output's gradient into one matrix:

$$J = \frac{\partial \mathbf{f}}{\partial \mathbf{x}} =
\begin{pmatrix} \frac{\partial f_1}{\partial x_1} & \cdots & \frac{\partial f_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\ \frac{\partial f_m}{\partial x_1} & \cdots & \frac{\partial f_m}{\partial x_n}
\end{pmatrix} \in \mathbb{R}^{m\times n}$$

Row $i$ of $J$ is $\nabla f_i$ — the gradient of the $i$-th output. The Jacobian is the
*correct* multivariate generalization of "derivative": it's the matrix such that
$\mathbf{f}(\mathbf{x}+\mathbf{h}) \approx \mathbf{f}(\mathbf{x}) + J\mathbf{h}$, i.e. the best
*linear* approximation of $\mathbf{f}$ near $\mathbf{x}$, exactly mirroring $f(a+h)\approx
f(a)+f'(a)h$ from `02.02`. When $m=1$, $J$ is just $\nabla f$ written as a row vector.

### The chain rule for Jacobians

If $\mathbf{h}(\mathbf{x}) = \mathbf{g}(\mathbf{f}(\mathbf{x}))$, the multivariate chain rule
from `02.04` becomes a matrix product: $J_h(\mathbf{x}) = J_g(\mathbf{f}(\mathbf{x})) \cdot
J_f(\mathbf{x})$. Composing functions composes (multiplies) their Jacobians — this is precisely
how backpropagation through a stack of neural network layers works: each layer's local
Jacobian gets multiplied into the running product as gradients flow backward.

### The Hessian matrix

For a scalar function $f:\mathbb{R}^n\to\mathbb{R}$, the **Hessian** is the matrix of all
second partial derivatives — the "Jacobian of the gradient":

$$H_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j}$$

By Clairaut's theorem (`02.03`), $H$ is symmetric ($H_{ij}=H_{ji}$) whenever $f$ is smooth
enough. $H$ describes **curvature**: its eigenvalues (recall `01.06`) tell you how the function
bends along each principal direction. If every eigenvalue is positive, $f$ curves upward in
every direction — a local minimum bowl. If eigenvalues have mixed signs, you're at a
**saddle point**: a minimum along some directions, a maximum along others (revisited in
`04.01`).

### Second-order Taylor expansion

Just as $f(a+h)\approx f(a)+f'(a)h$ is the first-order (linear) approximation, adding the next
term gives the second-order (quadratic) approximation, which for multivariate $f$ reads:

$$f(\mathbf{x}+\mathbf{h}) \approx f(\mathbf{x}) + \nabla f(\mathbf{x})^\top \mathbf{h} +
\frac{1}{2}\mathbf{h}^\top H(\mathbf{x}) \mathbf{h}$$

The gradient term captures the local slope; the Hessian term captures local curvature. This
quadratic model is exact for any function that's already a quadratic (like the loss surfaces
we've been plotting), and it's the foundation of Newton's method (`04.07`): rather than just
following the gradient downhill, jump straight to the minimum of this local quadratic
approximation.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Comparing a function's true surface against its first-order (linear) and second-order
(quadratic) Taylor approximations at a point shows visually how much curvature information
the Hessian adds: the quadratic model hugs the true surface much longer than the linear one.

In [ ]:
# f(x,y) = x^4 - 2x^2 + y^2 has a saddle-ish landscape; compare true surface vs 1st- and
# 2nd-order Taylor approximations at a point, as a 1D slice through y=y0 for readability.
f = lambda x, y: x**4 - 2*x**2 + y**2
grad_f = lambda x, y: np.array([4*x**3 - 4*x, 2*y])
hess_f = lambda x, y: np.array([[12*x**2 - 4, 0], [0, 2]])

x0, y0 = 1.3, 0.5
g, H = grad_f(x0, y0), hess_f(x0, y0)
x_line = np.linspace(x0 - 1.2, x0 + 1.2, 200)

true_vals = f(x_line, y0)
linear_approx = f(x0, y0) + g[0] * (x_line - x0)
quad_approx = f(x0, y0) + g[0] * (x_line - x0) + 0.5 * H[0, 0] * (x_line - x0) ** 2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(x_line, true_vals, color="#4C72B0", lw=2.5, label="true f(x, y=0.5)")
axes[0].plot(x_line, linear_approx, color="#DD8452", lw=1.5, ls="--", label="1st-order (linear)")
axes[0].plot(x_line, quad_approx, color="#55A868", lw=1.5, ls="--", label="2nd-order (quadratic)")
axes[0].plot(x0, f(x0, y0), 'ko', zorder=5)
axes[0].set_xlabel("x"); axes[0].set_ylabel("f(x, 0.5)")
axes[0].set_title("Quadratic (Hessian) approximation hugs the\ntrue curve far longer than linear")
axes[0].legend(fontsize=9)

# Eigenvalues of the Hessian at a few points -- classify curvature type at each
points = [(1.3, 0.5), (0.0, 0.0), (-1.0, 0.0)]
for x_p, y_p in points:
    eigvals = np.linalg.eigvalsh(hess_f(x_p, y_p))
    kind = "min-like (all +)" if np.all(eigvals > 0) else ("max-like (all -)" if np.all(eigvals < 0) else "SADDLE (mixed signs)")
    print(f"Point ({x_p:+.1f}, {y_p:+.1f}): Hessian eigenvalues = {np.round(eigvals, 2)}  -> {kind}")

x_grid = np.linspace(-1.8, 1.8, 200); y_grid = np.linspace(-1.5, 1.5, 200)
Xg, Yg = np.meshgrid(x_grid, y_grid)
axes[1].contour(Xg, Yg, f(Xg, Yg), levels=20, cmap="viridis")
for x_p, y_p in points:
    axes[1].plot(x_p, y_p, 'o', markersize=8)
    axes[1].annotate(f"({x_p},{y_p})", (x_p, y_p), textcoords="offset points", xytext=(8, 8))
axes[1].set_title("f(x,y)=x^4-2x^2+y^2: two minima + one saddle at origin")
axes[1].set_xlabel("x"); axes[1].set_ylabel("y")
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

We compute Jacobians and Hessians numerically from their definitions (nudge each coordinate,
per output/per pair of coordinates), verify the Jacobian chain rule on a composed function, and
confirm Hessian symmetry.

In [ ]:
def jacobian(f_vec, x, h=1e-5):
    """Numeric Jacobian of a vector-valued function f_vec: R^n -> R^m at point x."""
    x = np.array(x, dtype=float)
    f0 = np.array(f_vec(x))
    J = np.zeros((len(f0), len(x)))
    for j in range(len(x)):
        dx = np.zeros_like(x); dx[j] = h
        J[:, j] = (np.array(f_vec(x + dx)) - np.array(f_vec(x - dx))) / (2 * h)
    return J

def hessian(f_scalar, x, h=1e-4):
    """Numeric Hessian of a scalar function via finite differences of the gradient --
    literally 'the Jacobian of the gradient', matching the Theory section's framing."""
    n = len(x)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            xpp, xpm, xmp, xmm = [np.array(x, dtype=float) for _ in range(4)]
            xpp[i] += h; xpp[j] += h
            xpm[i] += h; xpm[j] -= h
            xmp[i] -= h; xmp[j] += h
            xmm[i] -= h; xmm[j] -= h
            H[i, j] = (f_scalar(xpp) - f_scalar(xpm) - f_scalar(xmp) + f_scalar(xmm)) / (4 * h * h)
    return H

# Jacobian chain rule check: h(x) = g(f(x)) where f: R^2->R^2, g: R^2->R^2
f_vec = lambda x: np.array([x[0]**2, x[0] * x[1]])
g_vec = lambda y: np.array([np.sin(y[0]), y[1]**2])
h_vec = lambda x: g_vec(f_vec(x))

x0 = np.array([1.2, 0.7])
J_f = jacobian(f_vec, x0)
J_g = jacobian(g_vec, f_vec(x0))
J_h_composed = J_g @ J_f              # chain rule: J_h = J_g(f(x)) . J_f(x)
J_h_direct = jacobian(h_vec, x0)      # computed directly on the composed function

print("Jacobian of f at x0:\n", np.round(J_f, 4))
print("\nJ_h via chain rule (J_g @ J_f):\n", np.round(J_h_composed, 4))
print("\nJ_h computed directly on h=g(f(x)):\n", np.round(J_h_direct, 4))
print("\nMatch:", np.allclose(J_h_composed, J_h_direct, atol=1e-3))

# Hessian symmetry check on a non-trivial function
f_scalar = lambda x: x[0]**3 * x[1]**2 + np.exp(x[0] * x[1])
H = hessian(f_scalar, np.array([0.6, 0.9]))
print("\nHessian:\n", np.round(H, 4))
print("Symmetric (H == H.T):", np.allclose(H, H.T, atol=1e-2))
print("Eigenvalues:", np.round(np.linalg.eigvalsh(H), 3))

## 🤖 Why This Matters for AI

Every layer in a neural network is a vector-valued function, and PyTorch's backward pass is
literally the Jacobian chain rule from this notebook: each layer contributes its local
Jacobian, and reverse-mode autodiff multiplies them together (technically, multiplies
Jacobian-vector products, to avoid ever forming a full Jacobian matrix — full Jacobians for a
layer with millions of units would be far too large to store). The Hessian shows up wherever
optimization needs curvature: **second-order methods** (`04.07`) use it directly, and the
*sharpness* of minima found by SGD (flat vs. sharp, tied to Hessian eigenvalues) is an active
area of generalization research (`04.09`). Below, we use PyTorch's `functorch`-style
`torch.autograd.functional` utilities to compute a real Jacobian and Hessian for a small
network, and confirm they match our from-scratch numeric versions.

In [ ]:
import torch
from torch.autograd.functional import jacobian as torch_jacobian
from torch.autograd.functional import hessian as torch_hessian

# 1) Confirm torch.autograd's Jacobian matches our from-scratch numeric one, on the SAME f
def f_vec_torch(x):
    return torch.stack([x[0] ** 2, x[0] * x[1]])

x0_t = torch.tensor([1.2, 0.7])
J_torch = torch_jacobian(f_vec_torch, x0_t).numpy()
print("Jacobian (torch.autograd):\n", np.round(J_torch, 4))
print("Jacobian (our from-scratch):\n", np.round(J_f, 4))
print("Match:", np.allclose(J_torch, J_f, atol=1e-3))

# 2) Confirm torch.autograd's Hessian matches ours, and use its eigenvalues to classify a
#    small "loss landscape" the way an optimizer would check for a saddle vs. a true minimum
def f_scalar_torch(x):
    return x[0] ** 3 * x[1] ** 2 + torch.exp(x[0] * x[1])

x1_t = torch.tensor([0.6, 0.9])
H_torch = torch_hessian(f_scalar_torch, x1_t).numpy()
print("\nHessian (torch.autograd):\n", np.round(H_torch, 4))
print("Hessian (our from-scratch):\n", np.round(H, 4))
print("Match:", np.allclose(H_torch, H, atol=1e-2))

eigvals = np.linalg.eigvalsh(H_torch)
print(f"\nEigenvalues: {np.round(eigvals, 3)}")
if np.all(eigvals > 0):
    print("All positive -> this point sits in a convex (bowl-shaped) region: a local minimum.")
elif np.all(eigvals < 0):
    print("All negative -> local maximum.")
else:
    print("Mixed signs -> SADDLE POINT: Newton's method without care would step toward it, not away.")

## 🏋️ Exercises

### Warm-up
1. For $\mathbf{f}(x,y) = (x^2y,\ xy^2)$, compute the $2\times2$ Jacobian by hand at
   $(x,y)=(1,2)$. Verify with `jacobian`.

### Practice
2. For $f(x,y) = x^2 + y^4$, compute the Hessian by hand as a function of $(x,y)$, then
   evaluate it numerically at $(0,0)$ and $(1,1)$ with `hessian`. Explain why the eigenvalues
   at $(0,0)$ don't fully determine whether it's a minimum (hint: what does the *quartic*
   $y^4$ term do that a Hessian, which only captures up-to-quadratic behavior, can't see?).

### Challenge
3. Build a 2-layer `torch` network (`Linear(3,4) -> tanh -> Linear(4,1)`) and use
   `torch.autograd.functional.hessian` to compute the Hessian of its scalar output with
   respect to its 3 *inputs* (not its weights) at a random input point. Report the
   eigenvalues and classify the point as min-like, max-like, or a saddle. Then explain why,
   for a real network with millions of *parameters* rather than 3 inputs, forming this full
   Hessian matrix (which would be millions-by-millions) is intractable in practice.

---

## 📚 Further Reading
- Deisenroth, Faisal & Ong, *Mathematics for Machine Learning*, Ch. 5.6–5.9 (Jacobian, Hessian, Taylor Series)
- Martens, ["New Insights and Perspectives on the Natural Gradient Method"](https://arxiv.org/abs/1412.1193) (curvature-aware optimization)

---

*Next notebook: [Integration Fundamentals](07_integration_fundamentals.ipynb)*